[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/09_tag_3_5_initialisierungen.ipynb)

# Tag 3-5 - 09 Initialisierung

Initialisierung bestimmt die Startwerte der Gewichte. Gute Initialisierung hält Aktivierungen und Gradienten in einem brauchbaren Größenbereich. Wir nutzen hier den Breast-Cancer-Datensatz aus scikit-learn, weil er ein sauberer binärer Tabellendatensatz mit echten Feature-Namen ist.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


In [ ]:
data = load_breast_cancer()
X = StandardScaler().fit_transform(data.data).astype("float32")
y = data.target.astype("float32")
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
)
print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)


In [ ]:
initializer_factories = {
    "RandomNormal": lambda: keras.initializers.RandomNormal(mean=0.0, stddev=1.0),
    "GlorotUniform": lambda: keras.initializers.GlorotUniform(),
    "HeNormal": lambda: keras.initializers.HeNormal(),
    "Orthogonal": lambda: keras.initializers.Orthogonal(gain=np.sqrt(2.0)),
}


def build_classifier(initializer_factory):
    model = keras.Sequential(
        [
            keras.Input(shape=(X_train.shape[1],), name="features"),
            # Parameter im Fokus: kernel_initializer setzt die Startgewichte.
            layers.Dense(32, activation="relu", kernel_initializer=initializer_factory(), name="first_dense"),
            layers.Dense(16, activation="relu", kernel_initializer=initializer_factory(), name="second_dense"),
            layers.Dense(1, activation="sigmoid", name="output"),
        ]
    )
    model.compile(optimizer=keras.optimizers.Adam(0.005), loss="binary_crossentropy", metrics=["accuracy"])
    return model


## Heatmaps der Startgewichte im ersten Layer


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for ax, (name, factory) in zip(axes, initializer_factories.items()):
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_classifier(factory)
    first_weights = model.get_layer("first_dense").get_weights()[0]
    im = ax.imshow(first_weights, aspect="auto", cmap="coolwarm")
    ax.set_title(name)
    ax.set_xlabel("Neuron im ersten Layer")
    ax.set_ylabel("Input-Feature")
    ax.set_yticks(range(0, len(feature_names), 5))
    ax.set_yticklabels(feature_names[::5], fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## Training mit denselben Hyperparametern

Jetzt wird jede Initialisierung mit demselben Modell, derselben Lernrate und derselben Epochenzahl trainiert. Im Loss-Plot nutzen wir eine logarithmische y-Achse, damit frühe Unterschiede sichtbar bleiben.


In [ ]:
histories = {}
models = {}
scores = {}
for name, factory in initializer_factories.items():
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_classifier(factory)
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=60,
        batch_size=32,
        verbose=0,
    )
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    histories[name] = history
    models[name] = model
    scores[name] = test_accuracy
    print(f"{name:>13}: Test Loss={test_loss:.3f}, Test Accuracy={test_accuracy:.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    axes[0].plot(history.history["val_loss"], label=name)
    axes[1].plot(history.history["val_accuracy"], label=name)
axes[0].set_yscale("log")
axes[0].set_title("Validation Loss (log scale)")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
plt.tight_layout()
plt.show()


## Confusion Matrix des besten Laufs

Die Confusion Matrix zeigt nicht nur, wie viele Beispiele richtig waren, sondern auch welche Fehlerarten auftreten.


In [ ]:
best_name = max(scores, key=scores.get)
best_model = models[best_name]
proba = best_model.predict(X_test, verbose=0).ravel()
y_pred = (proba >= 0.5).astype(int)
cm = confusion_matrix(y_test.astype(int), y_pred)

plt.figure(figsize=(4.5, 4))
plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
plt.xticks([0, 1], data.target_names)
plt.yticks([0, 1], data.target_names)
plt.xlabel("Vorhersage")
plt.ylabel("Wahrheit")
plt.title(f"Confusion Matrix: {best_name}")
plt.colorbar()
plt.show()
